# PINN SoH prediction with confidence intervals — all EVE cells with required inputs

For every EVE cell that has both **(a)** the three required `x_health` inputs (temperature, C-rate, DCIR in mΩ) and **(b)** measured SoH ground truth, this notebook:

1. Loads the fine-tuned PINN (`outputs/models/pinn_finetuned.pt`).
2. Pulls the cell-specific DCIR median from [`dcir_hppc_pulses.parquet`](../data/processed/dcir_hppc_pulses.parquet).
3. Runs `predict_rul_with_uncertainty` (200 MC-dropout samples + feature-noise perturbations) to get the **mean SoH trajectory** and the **5th–95th-percentile band**.
4. Evaluates the prediction at the *measured* cycle indices and reports MAE, RMSE, max-abs-error.
5. Renders one subplot per cell: measured points + predicted mean + 90 % CI band.

Caveats embedded in the model from the prior audit:
- Temperature is fixed at **25 °C** (the model was only trained at the isothermal-cohort temperature).
- C-rate is set to **0.5 C** for all cells (no per-cell C-rate available in the current pipeline).
- DCIR is the *only* cell-specific input that actually varies, in addition to the starting SoH.

In [ ]:
from __future__ import annotations
import os, sys
from pathlib import Path

HERE = Path.cwd()
ROOT = HERE.parent if (HERE.parent / 'CLAUDE.md').exists() else HERE
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display

from src.inference.predict_rul import (
    load_model, predict_rul_with_uncertainty, _load_measured_soh,
)

# Reproducibility for MC sampling
np.random.seed(456)
torch.manual_seed(456)

MODEL_CKPT = Path('outputs/models/pinn_finetuned.pt')
model = load_model(ckpt_path=MODEL_CKPT)
print(f'Model: {MODEL_CKPT.name}, {model.n_parameters():,} params, EOL threshold = {model.eol}')

## 1. Identify cells with required inputs

Inputs needed for prediction:
- `temperature_C` — 25 °C for the isothermal EVE cohort.
- `c_rate` — set to 0.5 C (the training-time proxy; same value the fine-tune used for every cell).
- `dcir_mOhm` — per-cell median R₀ from the 1RC HPPC discharge fits (parquet has cells 0005-0008 only).
- `soh_0` — first measured SoH (typically 1.0 from the RPT cycle 1).
- Measured SoH ground truth for error scoring (RPT and/or Longterm).

In [ ]:
TEMPERATURE_C = 25.0
C_RATE        = 0.5

# 1RC discharge R0 per cell (cohort) from data/processed/dcir_hppc_pulses.parquet
pulses = pd.read_parquet('data/processed/dcir_hppc_pulses.parquet')
dcir_short = pulses[(pulses['direction'] == 'discharge') & (pulses['pulse_kind'] == 'short')]
dcir_per_cell = (dcir_short.groupby('cell_id')['R0_Ohm'].median() * 1000.0).round(3)
print('DCIR mΩ (median 1RC discharge short pulses) per cell:')
print(dcir_per_cell.to_string())

candidates = sorted(dcir_per_cell.index.astype(str).tolist())
ready = []
for cid in candidates:
    measured = _load_measured_soh(cid)
    if not measured:
        continue
    merged = pd.concat(measured.values(), ignore_index=True).dropna(subset=['SOH'])
    merged = merged.sort_values('cycle_n').drop_duplicates('cycle_n').reset_index(drop=True)
    if len(merged) < 3:
        continue
    ready.append({
        'cell': cid,
        'T_C': TEMPERATURE_C,
        'c_rate': C_RATE,
        'dcir_mOhm': float(dcir_per_cell.loc[cid]),
        'n_obs': len(merged),
        'cycle_max_measured': int(merged['cycle_n'].max()),
        'soh_start': float(merged['SOH'].iloc[0]),
        'soh_end':   float(merged['SOH'].iloc[-1]),
    })
ready_df = pd.DataFrame(ready)
print(f'\n{len(ready_df)} cells with required inputs + ground truth:')
display(ready_df.round(4))

## 2. Predict per cell with MC-dropout uncertainty

For each ready cell we call `predict_rul_with_uncertainty` with 200 MC samples. The function:
- Adds ±1 % Gaussian noise to the input feature vector on each draw.
- Enables dropout in the ODE network so each draw yields a slightly different trajectory.
- Returns mean, 5th-percentile, and 95th-percentile SoH at every cycle in the prediction horizon → the **shaded band on the plots is a 90 % CI**.

Prediction horizon: from cycle 0 to either **(i)** the EOL crossing or **(ii)** `max_cycles = 2000`, whichever comes first.

In [ ]:
MAX_CYCLES_HORIZON = 2000
N_MC = 200

predictions = {}
for _, r in ready_df.iterrows():
    cid = r['cell']
    x_health = np.array([r['T_C'], r['c_rate'], r['dcir_mOhm'], 0.0, 1.0], dtype=np.float32)
    out = predict_rul_with_uncertainty(
        model=model,
        soh_now=float(r['soh_start']),
        cycle_now=0.0,
        x_health=x_health,
        n_samples=N_MC,
        feature_noise_std=0.01,
        max_cycles=MAX_CYCLES_HORIZON,
        n_points=400,
    )
    predictions[cid] = out
    print(f'  cell {cid}: RUL mean={out["rul_mean"]:.0f} cyc  P5={out["rul_p5"]:.0f}  P95={out["rul_p95"]:.0f}  σ={out["rul_std"]:.0f}')

## 3. Evaluate prediction error vs measured SoH

Interpolate the mean predicted trajectory to the measured cycle indices and compute residuals. Also count how many measured points lie *inside* the 90 % CI band — a calibration check (target ≈ 90 %).

In [ ]:
scoring = []
for _, r in ready_df.iterrows():
    cid = r['cell']
    out = predictions[cid]
    n_axis = np.array(out['n_axis'])
    s_mean = np.array(out['soh_trajectory_mean'])
    s_p5   = np.array(out['soh_trajectory_p5'])
    s_p95  = np.array(out['soh_trajectory_p95'])

    meas = pd.concat(_load_measured_soh(cid).values(), ignore_index=True)
    meas = meas.dropna(subset=['SOH']).sort_values('cycle_n').drop_duplicates('cycle_n').reset_index(drop=True)
    cycles = meas['cycle_n'].astype(float).to_numpy()
    soh_m  = meas['SOH'].astype(float).to_numpy()

    s_mean_at  = np.interp(cycles, n_axis, s_mean)
    s_p5_at    = np.interp(cycles, n_axis, s_p5)
    s_p95_at   = np.interp(cycles, n_axis, s_p95)
    err = s_mean_at - soh_m
    inside = ((soh_m >= s_p5_at) & (soh_m <= s_p95_at)).mean()

    scoring.append({
        'cell': cid,
        'n_obs': len(cycles),
        'mae_%SoH':  round(float(np.mean(np.abs(err)))*100, 3),
        'rmse_%SoH': round(float(np.sqrt(np.mean(err**2)))*100, 3),
        'max_abs_%SoH': round(float(np.max(np.abs(err)))*100, 3),
        'bias_%SoH (mean signed err)': round(float(np.mean(err))*100, 3),
        'CI_coverage_%': round(float(inside)*100, 1),
        'rul_mean_cyc':  round(out['rul_mean'], 0),
        'rul_p5_cyc':    round(out['rul_p5'], 0),
        'rul_p95_cyc':   round(out['rul_p95'], 0),
    })
scoring_df = pd.DataFrame(scoring)
scoring_df.to_csv('outputs/results/pinn_predictions_with_ci.csv', index=False)
print('Per-cell error + CI coverage:')
display(scoring_df)

print('\nAggregate over cells:')
print(f'  mean MAE:       {scoring_df["mae_%SoH"].mean():.3f} %SoH')
print(f'  mean RMSE:      {scoring_df["rmse_%SoH"].mean():.3f} %SoH')
print(f'  mean CI cover:  {scoring_df["CI_coverage_%"].mean():.1f} %  (target ≈ 90 %)')
print(f'  RUL spread:     P5={scoring_df["rul_p5_cyc"].min():.0f}-{scoring_df["rul_p5_cyc"].max():.0f},  '
      f'P95={scoring_df["rul_p95_cyc"].min():.0f}-{scoring_df["rul_p95_cyc"].max():.0f}')

## 4. Per-cell trajectory plots — one figure per cell

Each cell gets its own full-width subplot. Black dots are the measured SoH points; the red line is the PINN mean trajectory; the red shaded band is the 90 % CI (P5–P95) from the MC-dropout draws. Horizontal dashed line marks the EOL threshold (0.80 SoH); vertical dotted line marks the cycle where the mean trajectory crosses it.

Individual PNGs are also written to `outputs/results/pinn_per_cell/` so they can be embedded elsewhere.

In [ ]:
out_dir = Path('outputs/results/pinn_per_cell')
out_dir.mkdir(parents=True, exist_ok=True)

for _, r in ready_df.iterrows():
    cid = r['cell']
    out = predictions[cid]
    n_axis = np.array(out['n_axis'])
    s_mean = np.array(out['soh_trajectory_mean'])
    s_p5   = np.array(out['soh_trajectory_p5'])
    s_p95  = np.array(out['soh_trajectory_p95'])

    meas = pd.concat(_load_measured_soh(cid).values(), ignore_index=True)
    meas = meas.dropna(subset=['SOH']).sort_values('cycle_n').drop_duplicates('cycle_n').reset_index(drop=True)
    score_row = next(s for s in scoring if s['cell'] == cid)

    fig, ax = plt.subplots(figsize=(8.5, 5.0))
    ax.fill_between(n_axis, s_p5, s_p95, color='C3', alpha=0.18, label='90 % CI (P5–P95)')
    ax.plot(n_axis, s_mean, '-', color='C3', lw=1.6, label='PINN mean')
    ax.plot(meas['cycle_n'], meas['SOH'], 'ko', ms=4.5, alpha=0.85,
            label=f'measured ({len(meas)} points)')
    ax.axhline(model.eol, ls='--', color='gray', alpha=0.6, lw=1.0,
               label=f'EOL (SoH = {model.eol})')
    ax.axvline(out['rul_mean'], ls=':', color='C3', alpha=0.7, lw=1.0,
               label=f'mean EOL @ {int(out["rul_mean"])} cyc')

    ax.set(xlabel='Cycle', ylabel='SoH', ylim=(0.70, 1.04))
    title = (f'Cell {cid} — PINN SoH prediction with 90 % CI\n'
             f'DCIR = {r["dcir_mOhm"]:.2f} mΩ  |  '
             f'MAE = {score_row["mae_%SoH"]:.2f} %SoH, RMSE = {score_row["rmse_%SoH"]:.2f} %SoH, '
             f'max abs = {score_row["max_abs_%SoH"]:.2f} %SoH  |  '
             f'CI coverage = {score_row["CI_coverage_%"]:.0f} %')
    ax.set_title(title, fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9, loc='lower left')
    fig.tight_layout()

    out_path = out_dir / f'pinn_predict_cell_{cid}.png'
    fig.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'wrote → {out_path}')

print(f'\nScoring CSV → outputs/results/pinn_predictions_with_ci.csv')

## 5. Reading the results

- **MAE / RMSE** — prediction error in absolute SoH-percentage points. Battery characterisation noise is typically 0.2–0.5 %, so anything below 1 % is good.
- **Max abs error** — the worst single residual; reveals whether one outlier dominates.
- **Bias** — mean signed error; non-zero means the model systematically over- or under-predicts.
- **CI coverage %** — should sit close to 90 % for a well-calibrated 5–95 % interval. Much less → model is over-confident. Much more → too conservative.
- **RUL P5/P95** — the cycle number at which the 5th and 95th percentile trajectories first cross 0.80 SoH. These flank the mean RUL.

**Caveats** (the model is what it is):
- Only cells with significant fade (final SoH ≪ 1) give a meaningful test of degradation forecasting. The current EVE cohort has < 1.5 % fade over ≤ 73 measured cycles.
- DCIR is the only cell-specific differentiator in `x_health` here; cells with similar DCIR will produce nearly identical trajectories regardless of their measured fade rate.
- Confidence intervals widen with cycle number — the further from the training horizon, the less reliable.